In [ ]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os

load_dotenv(override=True)
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "localhost")
OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "ollama")

In [ ]:
def show(text):
	try:
		Console().print(text)
	except Exception:
		print(text)

In [ ]:
ollama = OpenAI(
   base_url=f"http://{OLLAMA_HOST}:{OLLAMA_PORT}/v1", 
   api_key=OPENAI_API_KEY # required but ignored
)
model_name = "llama3.1:8b"

In [ ]:
todos = []
completed = []

In [ ]:
def get_todo_report() -> str:
	result = ""
	for index, todo in enumerate(todos):
		if completed[index]:
			result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
		else:
			result += f"Todo #{index + 1}: {todo}\n"
	show(result)
	return result

In [ ]:
get_todo_report()

In [ ]:
def create_todos(descriptions: list[str]) -> str:
	todos.extend(descriptions)
	completed.extend([False] * len(descriptions))
	return get_todo_report()

In [ ]:
def mark_complete(index: int, completion_notes: str) -> str:
	if 1 <= index <= len(todos):
		completed[index - 1] = True
	else:
		return "No todo at this index."
	Console().print(completion_notes)
	return get_todo_report()

In [ ]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

In [ ]:
mark_complete(1, "bought")

In [ ]:
create_todos_json = {
	"name": "create_todos",
	"description": "Add new todos from a list of descriptions and return the full list",
	"parameters": {
		"type": "object",
		"properties": {
			"descriptions": {
					'type': 'array',
					'items': {'type': 'string'},
					'title': 'Descriptions'
					}
			},
		"required": ["descriptions"],
		"additionalProperties": False
	}
}

In [ ]:
mark_complete_json = {
	"name": "mark_complete",
	"description": "Mark complete the todo at the given position (starting from 1) and return the full list",
	"parameters": {
		'properties': {
			'index': {
					'description': 'The 1-based index of the todo to mark as complete',
					'title': 'Index',
					'type': 'integer'
					},
			'completion_notes': {
					'description': 'Notes about how you completed the todo in rich console markup',
					'title': 'Completion Notes',
					'type': 'string'
					}
			},
		'required': ['index', 'completion_notes'],
		'type': 'object',
		'additionalProperties': False
	}
}

In [ ]:

tools = [
   {"type": "function", "function": create_todos_json},
	{"type": "function", "function": mark_complete_json}
]

In [ ]:
def handle_tool_calls(tool_calls):
	results = []
	for tool_call in tool_calls:
		tool_name = tool_call.function.name
		arguments = json.loads(tool_call.function.arguments)
		tool = globals().get(tool_name)
		result = tool(**arguments) if tool else {}
		results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
	return results

In [ ]:
def loop(messages):
	done = False
	while not done:
		response = ollama.chat.completions.create(model=model_name, messages=messages, tools=tools, reasoning_effort="none")
		finish_reason = response.choices[0].finish_reason
		if finish_reason=="tool_calls":
			message = response.choices[0].message
			tool_calls = message.tool_calls
			results = handle_tool_calls(tool_calls)
			messages.append(message)
			messages.extend(results)
		else:
			done = True
	show(response.choices[0].message.content)

In [ ]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
todos, completed = [], []
loop(messages)